In [1]:
!pip install -r ../requirements.txt

In [2]:
import ollama
from ollama import Client


ollama_host = "http://host.docker.internal:11434"


client = Client(
  host=ollama_host,
)

response = client.chat(
    model='phi4',
    messages=[
        {
            'role': 'user',
            'content': 'Why is the sky blue?',
        },
    ]
)

print(response['message']['content'])


The sky appears blue due to a phenomenon called Rayleigh scattering. This occurs because molecules and small particles in Earth's atmosphere scatter sunlight in all directions. Sunlight, or white light, is composed of different colors, each with distinct wavelengths. Blue light has shorter wavelengths compared to other colors like red and yellow.

As sunlight enters the atmosphere, it collides with air molecules and tiny particles. Shorter wavelengths (blue and violet) are scattered more than longer wavelengths (red and yellow). However, our eyes are more sensitive to blue light and some of the violet light is absorbed by the upper atmosphere, which makes the sky appear predominantly blue during the day.

This scattering effect also explains why sunsets often appear red or orange. When the sun is near the horizon, its light has to pass through a greater thickness of Earth's atmosphere. The increased distance causes more scattering of shorter wavelengths and allows longer wavelengths li

In [10]:
corpus_json = "../dataset/mstro/corpus.jsonl"
cancer_types_json = "../resources/iknl_level_2_cancertypes.json"

import json

with open(corpus_json, 'r') as f:
    corpus_data = [json.loads(line) for line in f]

with open(cancer_types_json, 'r') as f:
    cancer_types_data = json.load(f)

cancer_types_data

[{'description': 'Lip', 'code': '101000'},
 {'description': 'Oral cavity', 'code': '102000'},
 {'description': 'Pharynx', 'code': '103000'},
 {'description': 'Salivary glands', 'code': '104000'},
 {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'},
 {'description': 'Larynx', 'code': '106000'},
 {'description': 'Thyroid gland', 'code': '107000'},
 {'description': 'Esophagus', 'code': '108000'},
 {'description': 'Stomach', 'code': '109000'},
 {'description': 'Small intestine', 'code': '110000'},
 {'description': 'Colon', 'code': '111000'},
 {'description': 'Rectosigmoid and rectum', 'code': '112000'},
 {'description': 'Anus and anal canal', 'code': '113000'},
 {'description': 'Liver and intrahepatic bile ducts', 'code': '114000'},
 {'description': 'Extrahepatic bile ducts and gallbladder', 'code': '115000'},
 {'description': 'Pancreas', 'code': '116000'},
 {'description': 'Nose', 'code': '117000'},
 {'description': 'Lungs and bronchus', 'code': '118000'},
 {'descript

In [5]:
# Create the prompt
def create_prompt(cancer_types_data, title, description):
    prompt = (
        "## Context: Cancer Type Identification from Clinical Study Information\n"
        "You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.\n\n"
        "## Task: Extract Cancer Type\n"
        "Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:\n"
        "{\n"
        '  "cancer_types": {[<list_of_detected_cancer_types>]\}n'
        "}\n"
        "If no clear match is found, return an empty list for \"cancer_types\".\n\n"
        "## Cancer Type List:\n"
    )
    prompt += f"{cancer_types_data}\n\n"
    prompt += (
        "## Input:\n"
        f"Title: {title}\n"
        f"Text: {description}\n"
    )
    return prompt

test_prompt = create_prompt(
    cancer_types_data,
    title="A Study on the Efficacy of Drug X in Treating Breast Cancer",
    description="This study investigates the effects of Drug X on patients diagnosed with breast cancer. The results show significant improvement in tumor reduction."
)
print(test_prompt)

## Context: Cancer Type Identification from Clinical Study Information
You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.

## Task: Extract Cancer Type
Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:
{
  "cancer_types": {[<list_of_detected_cancer_types>]\}n}
If no clear match is found, return an empty list for "cancer_types".

## Cancer Type List:
[{'description': 'Lip', 'code': '101000'}, {'description': 'Oral cavity', 'code': '102000'}, {'description': 'Pharynx', 'code': '103000'}, {'description': 'Salivary glands', 'code': '104000'}, {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'}, {'description': 'Larynx', 'code': '106000'}, {'descript

In [6]:
from tqdm import tqdm
import pandas as pd


# Process JSON transcription data and save intermediate results to Excel
def extract_cancer_types_for_corpus(client, model, cancer_types_data, corpus_data, excel_output_path):
    results = []

    for item in tqdm(corpus_data, desc=f"Extracting cancer_type using {model}"):
        prompt = create_prompt(cancer_types_data, item['title'], item['text'])
        # Use the model to generate a respons   
        print(f"Prompt: {prompt}")

        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.2
        ).choices[0].message.content.strip()

        error = None
        cancer_types = []

        # Attempt to parse the response as JSON
        try:
            # Remove markdown backticks if present
            response_cleaned = response.strip().strip('`').replace('json', '').strip()
            json_result = json.loads(response_cleaned)

            # Get the list of metaphors
            cancer_types = json_result.get('cancer_types', [])
            
        except (json.JSONDecodeError, TypeError, AttributeError) as e:
            error = f"Parsing error: {str(e)} | Response: {response}"

        # Store the result with metaphors as a serialized JSON string
        results.append({
            'cancer_types': json.dumps(cancer_types, ensure_ascii=False),
            'model': model,
            'error': error
        })

        # Save intermediate results to Excel after each item
        df = pd.DataFrame(results)
        df.to_excel(excel_output_path, index=False)

    return results

In [ ]:
from tqdm import tqdm
import pandas as pd


# Process JSON transcription data and save intermediate results to Excel
def extract_cancer_types_for_queries(client, model, cancer_types_data, queries, excel_output_path):
    results = []

    for item in tqdm(corpus_data, desc=f"Extracting cancer_type using {model}"):
        prompt = create_prompt(cancer_types_data, item['title'], item['text'])
        # Use the model to generate a respons   
        print(f"Prompt: {prompt}")

        messages = [{"role": "user", "content": prompt}]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.2
        ).choices[0].message.content.strip()

        error = None
        cancer_types = []

        # Attempt to parse the response as JSON
        try:
            # Remove markdown backticks if present
            response_cleaned = response.strip().strip('`').replace('json', '').strip()
            json_result = json.loads(response_cleaned)

            # Get the list of metaphors
            cancer_types = json_result.get('cancer_types', [])
            
        except (json.JSONDecodeError, TypeError, AttributeError) as e:
            error = f"Parsing error: {str(e)} | Response: {response}"

        # Store the result with metaphors as a serialized JSON string
        results.append({
            'cancer_types': json.dumps(cancer_types, ensure_ascii=False),
            'model': model,
            'error': error
        })

        # Save intermediate results to Excel after each item
        df = pd.DataFrame(results)
        df.to_excel(excel_output_path, index=False)

    return results

In [7]:
import aisuite as ai
import os

output_path = "../dataset/mstro/"


ollama_host = "http://host.docker.internal:11434"
os.environ["OLLAMA_API_URL"] = ollama_host

client = ai.Client()

model = "ollama:gemma3:12b"  # or any other model you want to use
model_name = model.replace(":", "_").replace("/", "_")
excel_output_path = f"{output_path}/corpus_cancer_types{model_name}.xlsx"
extracted_cancer_types = extract_cancer_types_for_corpus(client, model, cancer_types_data, corpus_data, excel_output_path)

# Output results
print(json.dumps(extracted_cancer_types, indent=2, ensure_ascii=False))

df_export = pd.DataFrame(extracted_cancer_types)
df_export.to_excel(excel_output_path, index=False)
print(f"Results written to {excel_output_path}")

Extracting cancer_type using ollama:gemma3:12b:   0%|          | 0/19 [00:00<?, ?it/s]

Prompt: ## Context: Cancer Type Identification from Clinical Study Information
You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.

## Task: Extract Cancer Type
Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:
{
  "cancer_types": {[<list_of_detected_cancer_types>]\}n}
If no clear match is found, return an empty list for "cancer_types".

## Cancer Type List:
[{'description': 'Lip', 'code': '101000'}, {'description': 'Oral cavity', 'code': '102000'}, {'description': 'Pharynx', 'code': '103000'}, {'description': 'Salivary glands', 'code': '104000'}, {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'}, {'description': 'Larynx', 'code': '106000'}, {'

Extracting cancer_type using ollama:gemma3:12b:   5%|▌         | 1/19 [00:19<05:43, 19.06s/it]

Prompt: ## Context: Cancer Type Identification from Clinical Study Information
You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.

## Task: Extract Cancer Type
Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:
{
  "cancer_types": {[<list_of_detected_cancer_types>]\}n}
If no clear match is found, return an empty list for "cancer_types".

## Cancer Type List:
[{'description': 'Lip', 'code': '101000'}, {'description': 'Oral cavity', 'code': '102000'}, {'description': 'Pharynx', 'code': '103000'}, {'description': 'Salivary glands', 'code': '104000'}, {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'}, {'description': 'Larynx', 'code': '106000'}, {'

Extracting cancer_type using ollama:gemma3:12b:  11%|█         | 2/19 [00:20<02:30,  8.85s/it]

...

Prompt: ## Context: Cancer Type Identification from Clinical Study Information
You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.

## Task: Extract Cancer Type
Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:
{
  "cancer_types": {[<list_of_detected_cancer_types>]\}n}
If no clear match is found, return an empty list for "cancer_types".

## Cancer Type List:
[{'description': 'Lip', 'code': '101000'}, {'description': 'Oral cavity', 'code': '102000'}, {'description': 'Pharynx', 'code': '103000'}, {'description': 'Salivary glands', 'code': '104000'}, {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'}, {'description': 'Larynx', 'code': '106000'}, {'

Extracting cancer_type using ollama:gemma3:12b:  95%|█████████▍| 18/19 [00:51<00:01,  1.97s/it]

Prompt: ## Context: Cancer Type Identification from Clinical Study Information
You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its title and text description. You will use a predefined list of cancer types for consistency and accuracy.

## Task: Extract Cancer Type
Given the clinical study's title and text description, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:
{
  "cancer_types": {[<list_of_detected_cancer_types>]\}n}
If no clear match is found, return an empty list for "cancer_types".

## Cancer Type List:
[{'description': 'Lip', 'code': '101000'}, {'description': 'Oral cavity', 'code': '102000'}, {'description': 'Pharynx', 'code': '103000'}, {'description': 'Salivary glands', 'code': '104000'}, {'description': 'Nasal cavity and paranasal sinuses', 'code': '105000'}, {'description': 'Larynx', 'code': '106000'}, {'

Extracting cancer_type using ollama:gemma3:12b: 100%|██████████| 19/19 [00:53<00:00,  2.79s/it]

[
  {
    "cancer_types": "[{\"description\": \"Brain and cranial nerves\", \"code\": \"139000\"}]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[{\"description\": \"Brain and cranial nerves\", \"code\": \"139000\"}]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[{\"description\": \"Meninges\", \"code\": \"140000\"}]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[{\"description\": \"Brain and cranial nerves\", \"code\": \"139000\"}]",
    "model": "ollama:gemma3:12b",
    "error": null
  },
  {
    "cancer_types": "[{\"description\": \"Colon\", \"code\": \"111000\"}]",
    "model": "ollama:gemma3:12b",
    "error": nu

In [11]:
merged_data = []
for corpus, cancer_info in zip(corpus_data, extracted_cancer_types):
    merged_entry = corpus.copy()
    merged_entry["cancer_types"] = cancer_info.get("cancer_types", [])
    merged_data.append(merged_entry)


In [12]:
merged_data

[{'_id': 'Ependymoom studie',
  'title': 'Ependymoom studie',
  'text': 'Study Type: Observationeel, Indication: Registratiestudie voor ependymoom om tot een landelijk protocol te komen',
  'metadata': {'brief_title': 'Ependymoom studie',
   'phase': 'Open',
   'drugs': [],
   'drugs_list': [],
   'diseases': [],
   'diseases_list': [],
   'enrollment': 'Unknown',
   'inclusion_criteria': '- Diagnosis of ependymoma: diagnosis should be pathologically verified and be confirmed by central review whenever \npossible. Central review will be done by: J.M. Kros, neuropathologist\n\n\n\tMinimaal 18 jaar',
   'exclusion_criteria': '',
   'brief_summary': 'Study Type: Observationeel, Indication: Registratiestudie voor ependymoom om tot een landelijk protocol te komen'},
  'cancer_types': '[{"description": "Brain and cranial nerves", "code": "139000"}]'},
 {'_id': 'ERROR',
  'title': 'ERROR',
  'text': 'Study Type: Prospective, Indication: Effect of stress and exercise on the outcome after chemo

In [13]:
reports = "/home/jovyan/work/TrialGPT/dataset/mstro/queries.jsonl"
with open(reports, 'r') as f:
    report_data = [json.loads(line) for line in f]
report_data

[{'_id': '006_MDO_C_searchable.pdf',
  'text': "& zuyderland patiéntnummer 006\nAfz: Multidisciplinair overleg\nDatum: : 15.09.2021\nDocument datum: : 07.09.2021\nDocumentversie: : 00\nBetreft: ER o2b en 19.11.1953,\nSS\nGeachte collega,\nBovengenoemde patiént(e) is besproken tijdens ons multi disciplinair overleg thorax\noncologie.\nDatum bespreking: 06.09.2021\nBetreft: een nieuwe patient. sa\nHoofdklacht\nminder energie, iets meer dyspnoe\nWHO\n1\nMedische beeldvorming\nPET-CT: Conclusie:\n- Beeld sterk verdacht voor een longmaligniteit gelegen paramediastinaal in de bovenkwab\nvan de linkerlong; maximale diameter\ntenminste 10 cm; SUVmax 20; centrale necrose; ingroei in het mediastinum; doorgroei door\nde fissura major; betrokkenheid van de\nlinkerhilus en de bijbehorende vaten van de bovenkwab van de linkerlong, arcus aortae en\nproximale aorta descendens.\n- Tenminste één lymfekliermetastase in mediastinaal lymfeklierstation 5.\n- Geen aanwijzingen voor metastasen op afstand.\nMR

In [ ]:
# Create the prompt
def create_prompt_text(cancer_types_data, text):
    prompt = (
        "## Context: Cancer Type Identification from Clinical Study Information\n"
        "You are a clinical research data specialist tasked with identifying the primary cancer type associated with a clinical study based on its text description. You will use a predefined list of cancer types for consistency and accuracy.\n\n"
        "## Task: Extract Cancer Type\n"
        "Given the clinical study's text, identify the most relevant cancer type from the provided list. Respond with a valid JSON object that exactly follows this schema:\n"
        "{\n"
        '  "cancer_types": {[<list_of_detected_cancer_types>]\}n'
        "}\n"
        "If no clear match is found, return an empty list for \"cancer_types\".\n\n"
        "## Cancer Type List:\n"
    )
    prompt += f"{cancer_types_data}\n\n"
    prompt += (
        "## Input:\n"

        f"Text: {text}\n"
    )
    return prompt

test_prompt = create_prompt(
    cancer_types_data,
    title="A Study on the Efficacy of Drug X in Treating Breast Cancer",
    description="This study investigates the effects of Drug X on patients diagnosed with breast cancer. The results show significant improvement in tumor reduction."
)
print(test_prompt)